In [6]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/daisee/DAiSEE/README.txt
/kaggle/input/daisee/DAiSEE/hog.py
/kaggle/input/daisee/DAiSEE/extractFrames.py
/kaggle/input/daisee/DAiSEE/Labels/TrainLabels.csv
/kaggle/input/daisee/DAiSEE/Labels/AllLabels.csv
/kaggle/input/daisee/DAiSEE/Labels/TestLabels.csv
/kaggle/input/daisee/DAiSEE/Labels/ValidationLabels.csv
/kaggle/input/daisee/DAiSEE/DataSet/Test.txt
/kaggle/input/daisee/DAiSEE/DataSet/Validation.txt
/kaggle/input/daisee/DAiSEE/DataSet/Train.txt
/kaggle/input/daisee/DAiSEE/DataSet/Validation/400023/4000231047/4000231047.avi
/kaggle/input/daisee/DAiSEE/DataSet/Validation/400023/4000231001/4000231001.avi
/kaggle/input/daisee/DAiSEE/DataSet/Validation/400023/4000231049/4000231049.avi
/kaggle/input/daisee/DAiSEE/DataSet/Validation/400023/4000232035/4000232035.avi
/kaggle/input/daisee/DAiSEE/DataSet/Validation/400023/4000232044/4000232044.avi
/kaggle/input/daisee/DAiSEE/DataSet/Validation/400023/4000232042/4000232042.avi
/kaggle/input/daisee/DAiSEE/DataSet/Validation/400023

In [7]:
# Dataset is auto-available at:
dataset_path = '/kaggle/input/daisee'

import os
print("Dataset contents:")
print(os.listdir(dataset_path))

# Check structure
for folder in os.listdir(dataset_path):
    print(f"\n{folder}:")
    print(os.listdir(f'{dataset_path}/{folder}'))


Dataset contents:
['DAiSEE']

DAiSEE:
['Labels', 'DataSet', 'README.txt', 'GenderClips', 'hog.py', 'extractFrames.py']


In [4]:
# =============================================================================
# GranulationCNN — FULL baseline block (uint8 cache fix for Kaggle disk limit)
# Paste into ONE Kaggle cell. GPU: Settings -> Accelerator -> GPU T4.
# Caches frames as uint8 (~4.7GB instead of ~38GB), normalizes at load time.
# First run extracts frames ONCE (slow). Re-runs reuse cache (fast).
# =============================================================================
import os, re, cv2, shutil, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, cohen_kappa_score)

# ----------------------------- config ---------------------------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)

ROOT, TARGET, NUM_CLASSES = "/kaggle/input/daisee/DAiSEE", "Engagement", 4
VIDEOS_ROOT, LABELS_ROOT = f"{ROOT}/DataSet", f"{ROOT}/Labels"
T_FRAMES, G_GRANULES, LAMBDA = 8, 4, 0.3
BATCH, EPOCHS, LR = 16, 5, 3e-4
CACHE_DIR = "/kaggle/working/frames_cache"

# clear any half-written cache from a previous failed run, then recreate
shutil.rmtree(CACHE_DIR, ignore_errors=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print("cache dir ready:", CACHE_DIR)

# ----------------------------- labels + paths --------------------------------
def cid(x): return max(re.findall(r"\d+", str(x)), key=len)
def load_labels(p):
    df = pd.read_csv(p)
    return {cid(r["ClipID"]): int(r[TARGET]) for _, r in df.iterrows()}
labels = {"train": load_labels(f"{LABELS_ROOT}/TrainLabels.csv"),
          "val":   load_labels(f"{LABELS_ROOT}/ValidationLabels.csv"),
          "test":  load_labels(f"{LABELS_ROOT}/TestLabels.csv")}
def index_videos(split):
    out = {}
    for r, _, fs in os.walk(os.path.join(VIDEOS_ROOT, split)):
        for f in fs:
            if f.endswith(".avi"): out[cid(f)] = os.path.join(r, f)
    return out
videos = {"train": index_videos("Train"),
          "val":   index_videos("Validation"),
          "test":  index_videos("Test")}
pairs = {k: [(i, videos[k][i], labels[k][i]) for i in videos[k] if i in labels[k]]
         for k in videos}
print("pairs:", {k: len(v) for k, v in pairs.items()})

# ----------------------------- extract frames ONCE (uint8) -------------------
def extract_clip(path):
    cap = cv2.VideoCapture(path)
    if not cap.isOpened(): return None
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if n <= 0: cap.release(); return None
    want = set(np.linspace(0, n - 1, T_FRAMES).astype(int).tolist())
    grabbed, i = [], 0
    while len(grabbed) < T_FRAMES:
        if not cap.grab(): break
        if i in want:
            ok, fr = cap.retrieve()
            if ok:
                fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
                fr = cv2.resize(fr, (224, 224))
                grabbed.append(fr)                 # raw uint8 HWC
        i += 1
    cap.release()
    if not grabbed: return None
    while len(grabbed) < T_FRAMES: grabbed.append(grabbed[-1].copy())
    return np.stack(grabbed[:T_FRAMES]).astype(np.uint8)   # (T,224,224,3) uint8

def build_cache(split):
    out, bad = [], 0
    d = os.path.join(CACHE_DIR, split); os.makedirs(d, exist_ok=True)
    for k, (clip_id, path, y) in enumerate(pairs[split]):
        npy = os.path.join(d, f"{clip_id}.npy")
        if not os.path.exists(npy):
            arr = extract_clip(path)
            if arr is None: bad += 1; continue
            np.save(npy, arr)
        out.append((npy, y))
        if (k + 1) % 250 == 0: print(f"  [{split}] {k+1}/{len(pairs[split])} (bad={bad})")
    print(f"[{split}] {len(out)} cached, {bad} skipped")
    return out

cached = {s: build_cache(s) for s in ["train", "val", "test"]}

# ----------------------------- dataset / loaders (normalize on load) ---------
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class CachedDAiSEE(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        npy, y = self.items[idx]
        arr = np.load(npy)                                       # (T,224,224,3) uint8
        x = torch.from_numpy(arr).float().permute(0, 3, 1, 2) / 255.0
        x = (x - MEAN) / STD                                     # (T,3,224,224)
        return x, torch.tensor(y, dtype=torch.long)

train_loader = DataLoader(CachedDAiSEE(cached["train"]), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(CachedDAiSEE(cached["val"]),   batch_size=BATCH,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(CachedDAiSEE(cached["test"]),  batch_size=BATCH,
                          num_workers=2, pin_memory=True)

# ----------------------------- model -----------------------------------------
class GranulationCNN(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, NUM_CLASSES)
    def forward(self, x):
        B, T_, C, H, W = x.shape
        z = self.backbone(x.view(B * T_, C, H, W)).flatten(1).view(B, T_, -1)
        avg_logits = self.fc(z).mean(1)
        glogits, cons = [], 0.0
        for i in range(B):
            zi = z[i]
            assign = torch.randint(0, G_GRANULES, (T_,), device=zi.device)
            pl = []
            for g in range(G_GRANULES):
                m = assign == g
                if m.any():
                    proto = zi[m].mean(0)
                    pl.append(self.fc(proto))
                    cons += ((zi[m] - proto) ** 2).mean()
            glogits.append(torch.stack(pl).mean(0))
        return avg_logits, torch.stack(glogits), cons / B

# ----------------------------- train -----------------------------------------
def run_epoch(model, loader, opt=None, scaler=None):
    train = opt is not None
    model.train(train)
    tot, cor, loss_sum = 0, 0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast(enabled=(DEVICE == "cuda")):
                _, gl, cons = model(x)
                loss = F.cross_entropy(gl, y) + LAMBDA * cons
            if train:
                opt.zero_grad()
                if scaler:
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
            cor += (gl.argmax(1) == y).sum().item()
            tot += y.size(0); loss_sum += loss.item() * y.size(0)
    return loss_sum / tot, cor / tot

model  = GranulationCNN().to(DEVICE)
opt    = torch.optim.Adam(model.parameters(), lr=LR)
scaler = GradScaler(enabled=(DEVICE == "cuda"))
best = 0.0
for e in range(1, EPOCHS + 1):
    tr = run_epoch(model, train_loader, opt, scaler)
    va = run_epoch(model, val_loader)
    print(f"Epoch {e:02d} | TrainAcc={tr[1]:.4f} | ValAcc={va[1]:.4f}")
    if va[1] > best:
        best = va[1]; torch.save(model.state_dict(), "best_granulation.pt")
print("Best Val Acc:", best)

# ----------------------------- evaluate --------------------------------------
model.load_state_dict(torch.load("best_granulation.pt")); model.eval()
def evaluate(model, loader):
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            _, gl, _ = model(x)
            P.append(gl.argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)
P, Y = evaluate(model, test_loader)

acc = accuracy_score(Y, P)
pr, rc, f1, _ = precision_recall_fscore_support(Y, P, average="weighted", zero_division=0)
kappa = cohen_kappa_score(Y, P)
cm = confusion_matrix(Y, P)
print(f"\n[4-class] Acc={acc:.4f}  Prec(W)={pr:.4f}  Rec(W)={rc:.4f}  F1(W)={f1:.4f}  kappa={kappa:.4f}")
print("Confusion matrix (4-class):\n", cm)
p, r, f1pc, sup = precision_recall_fscore_support(Y, P, average=None, zero_division=0)
for c in range(NUM_CLASSES):
    print(f"  C{c}: P={p[c]:.4f} R={r[c]:.4f} F1={f1pc[c]:.4f} support={sup[c]}")

# ----------------------------- binary C2/C3 view -----------------------------
mask = (Y == 2) | (Y == 3)
Yb, Pb = Y[mask], P[mask]
accb = accuracy_score(Yb, Pb)
prb, rcb, f1b, _ = precision_recall_fscore_support(Yb, Pb, average="weighted", labels=[2,3], zero_division=0)
kappab = cohen_kappa_score(Yb, Pb)
cmb = confusion_matrix(Yb, Pb, labels=[2,3])
print(f"\n[binary C2/C3] n={int(mask.sum())}  Acc={accb:.4f}  Prec(W)={prb:.4f}  Rec(W)={rcb:.4f}  F1(W)={f1b:.4f}  kappa={kappab:.4f}")
print("Confusion matrix (binary, rows/cols = C2,C3):\n", cmb)

Device: cuda
cache dir ready: /kaggle/working/frames_cache
pairs: {'train': 4852, 'val': 1429, 'test': 1638}
  [train] 250/4852 (bad=0)
  [train] 500/4852 (bad=0)
  [train] 750/4852 (bad=0)
  [train] 1000/4852 (bad=0)
  [train] 1250/4852 (bad=0)


[mpeg4 @ 0x26f110c0] I cbpy damaged at 16 8
[mpeg4 @ 0x26f110c0] Error at MB: 344


  [train] 1500/4852 (bad=0)
  [train] 1750/4852 (bad=0)
  [train] 2000/4852 (bad=0)
  [train] 2250/4852 (bad=0)
  [train] 2500/4852 (bad=0)
  [train] 2750/4852 (bad=0)
  [train] 3000/4852 (bad=0)
  [train] 3250/4852 (bad=0)
  [train] 3500/4852 (bad=0)
  [train] 3750/4852 (bad=0)
  [train] 4000/4852 (bad=0)
  [train] 4250/4852 (bad=0)
  [train] 4500/4852 (bad=0)
  [train] 4750/4852 (bad=0)
[train] 4852 cached, 0 skipped
  [val] 250/1429 (bad=0)
  [val] 500/1429 (bad=0)
  [val] 750/1429 (bad=0)
  [val] 1000/1429 (bad=0)
  [val] 1250/1429 (bad=0)
[val] 1429 cached, 0 skipped
  [test] 250/1638 (bad=0)
  [test] 500/1638 (bad=0)
  [test] 750/1638 (bad=0)
  [test] 1000/1638 (bad=0)
  [test] 1250/1638 (bad=0)
  [test] 1500/1638 (bad=0)
[test] 1638 cached, 0 skipped


/tmp/ipykernel_57/2358968620.py:160: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE == "cuda"))
/tmp/ipykernel_57/2358968620.py:145: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE == "cuda")):


Epoch 01 | TrainAcc=0.5286 | ValAcc=0.3450
Epoch 02 | TrainAcc=0.5674 | ValAcc=0.4052
Epoch 03 | TrainAcc=0.5798 | ValAcc=0.5360
Epoch 04 | TrainAcc=0.6000 | ValAcc=0.5647
Epoch 05 | TrainAcc=0.5942 | ValAcc=0.5199
Best Val Acc: 0.5647305808257522

[4-class] Acc=0.5214  Prec(W)=0.4886  Rec(W)=0.5214  F1(W)=0.3900  kappa=0.0140
Confusion matrix (4-class):
 [[  0   0   4   0]
 [  0   0  77   4]
 [  0   0 819  30]
 [  0   0 669  35]]
  C0: P=0.0000 R=0.0000 F1=0.0000 support=4
  C1: P=0.0000 R=0.0000 F1=0.0000 support=81
  C2: P=0.5220 R=0.9647 F1=0.6774 support=849
  C3: P=0.5072 R=0.0497 F1=0.0906 support=704

[binary C2/C3] n=1553  Acc=0.5499  Prec(W)=0.5450  Rec(W)=0.5499  F1(W)=0.4244  kappa=0.0156
Confusion matrix (binary, rows/cols = C2,C3):
 [[819  30]
 [669  35]]


In [3]:
# =============================================================================
# GranulationCNN — VARIANT B: pretrained backbone + focal loss
# Paste into a NEW Kaggle cell AFTER the baseline cell has finished.
# Reuses the uint8 .npy frame cache, normalizes on load — trains in minutes.
# =============================================================================
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, cohen_kappa_score)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

NUM_CLASSES = 4
T_FRAMES, G_GRANULES, LAMBDA = 8, 4, 0.3
BATCH, EPOCHS, LR = 16, 8, 3e-4         # a couple more epochs; pretrained converges fast
CACHE_DIR = "/kaggle/working/frames_cache"

# ---- rebuild the cached item lists from disk --------------------------------
def list_cache(split):
    d = os.path.join(CACHE_DIR, split)
    items = []
    if "pairs" in globals():
        lab = {cid_: y for (cid_, _p, y) in pairs[split]}
    else:
        raise ValueError("Run the baseline cell first to load 'pairs' into memory!")
    
    if os.path.exists(d):
        for f in os.listdir(d):
            if f.endswith(".npy"):
                clip_id = f[:-4]
                if clip_id in lab:
                    items.append((os.path.join(d, f), lab[clip_id]))
    return items

cached_b = {s: list_cache(s) for s in ["train", "val", "test"]}
print("Cached items:", {k: len(v) for k, v in cached_b.items()})

# ---- dataset / loaders (normalize on load to match baseline) ----------------
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class CachedDAiSEE_Variant(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        npy, y = self.items[idx]
        arr = np.load(npy)                                       # (T,224,224,3) uint8
        x = torch.from_numpy(arr).float().permute(0, 3, 1, 2) / 255.0
        x = (x - MEAN) / STD                                     # (T,3,224,224)
        return x, torch.tensor(y, dtype=torch.long)

train_loader = DataLoader(CachedDAiSEE_Variant(cached_b["train"]), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(CachedDAiSEE_Variant(cached_b["val"]),   batch_size=BATCH,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(CachedDAiSEE_Variant(cached_b["test"]),  batch_size=BATCH,
                          num_workers=2, pin_memory=True)

# ---- class weights from TRAIN distribution ----------------------------------
train_labels = np.array([y for _, y in cached_b["train"]])
counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float32)
inv = counts.sum() / (counts + 1e-6)
class_w = torch.tensor(inv / inv.sum() * NUM_CLASSES, dtype=torch.float32, device=DEVICE)
print("train class counts:", counts.tolist())
print("class weights      :", class_w.cpu().numpy().round(3).tolist())

# ---- focal loss -------------------------------------------------------------
def focal_loss(logits, target, gamma=2.0, weight=None):
    logp = F.log_softmax(logits, dim=1)
    p = logp.exp()
    logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
    p_t = p.gather(1, target.unsqueeze(1)).squeeze(1)
    loss = -((1 - p_t) ** gamma) * logp_t
    if weight is not None:
        loss = loss * weight[target]
    return loss.mean()

# ---- model: PRETRAINED backbone --------------------------------------------
class GranulationCNN(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights="IMAGENET1K_V1")   # <-- pretrained
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, NUM_CLASSES)
    def forward(self, x):
        B, T_, C, H, W = x.shape
        z = self.backbone(x.view(B * T_, C, H, W)).flatten(1).view(B, T_, -1)
        avg_logits = self.fc(z).mean(1)
        glogits, cons = [], 0.0
        for i in range(B):
            zi = z[i]
            assign = torch.randint(0, G_GRANULES, (T_,), device=zi.device)
            pl = []
            for g in range(G_GRANULES):
                m = assign == g
                if m.any():
                    proto = zi[m].mean(0)
                    pl.append(self.fc(proto))
                    cons += ((zi[m] - proto) ** 2).mean()
            glogits.append(torch.stack(pl).mean(0))
        return avg_logits, torch.stack(glogits), cons / B

def run_epoch(model, loader, opt=None, scaler=None, use_focal=True):
    train = opt is not None
    model.train(train)
    tot, cor, loss_sum = 0, 0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast(enabled=(DEVICE == "cuda")):
                _, gl, cons = model(x)
                if use_focal:
                    cls = focal_loss(gl, y, gamma=2.0, weight=class_w)
                else:
                    cls = F.cross_entropy(gl, y, weight=class_w)
                loss = cls + LAMBDA * cons
            if train:
                opt.zero_grad()
                if scaler:
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
            cor += (gl.argmax(1) == y).sum().item()
            tot += y.size(0); loss_sum += loss.item() * y.size(0)
    return loss_sum / tot, cor / tot

model  = GranulationCNN().to(DEVICE)
opt    = torch.optim.Adam(model.parameters(), lr=LR)
scaler = GradScaler(enabled=(DEVICE == "cuda"))
best = 0.0
for e in range(1, EPOCHS + 1):
    tr = run_epoch(model, train_loader, opt, scaler, use_focal=True)
    va = run_epoch(model, val_loader, use_focal=True)
    print(f"Epoch {e:02d} | TrainAcc={tr[1]:.4f} | ValAcc={va[1]:.4f}")
    if va[1] > best:
        best = va[1]; torch.save(model.state_dict(), "best_granulation_pretrained_focal.pt")
print("Best Val Acc:", best)

# ---- evaluate ---------------------------------------------------------------
model.load_state_dict(torch.load("best_granulation_pretrained_focal.pt")); model.eval()
def evaluate(model, loader):
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            _, gl, _ = model(x)
            P.append(gl.argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)
P, Y = evaluate(model, test_loader)

acc = accuracy_score(Y, P)
pr, rc, f1, _ = precision_recall_fscore_support(Y, P, average="weighted", zero_division=0)
kappa = cohen_kappa_score(Y, P)
cm = confusion_matrix(Y, P)
print(f"\n[4-class] Acc={acc:.4f}  Prec(W)={pr:.4f}  Rec(W)={rc:.4f}  F1(W)={f1:.4f}  kappa={kappa:.4f}")
print("Confusion matrix (4-class):\n", cm)
p, r, f1pc, sup = precision_recall_fscore_support(Y, P, average=None, zero_division=0)
for c in range(NUM_CLASSES):
    print(f"  C{c}: P={p[c]:.4f} R={r[c]:.4f} F1={f1pc[c]:.4f} support={sup[c]}")

# ---- binary C2/C3 view (matches the paper's scope) --------------------------
mask = (Y == 2) | (Y == 3)
Yb, Pb = Y[mask], P[mask]
accb = accuracy_score(Yb, Pb)
prb, rcb, f1b, _ = precision_recall_fscore_support(Yb, Pb, average="weighted", labels=[2,3], zero_division=0)
kappab = cohen_kappa_score(Yb, Pb)
cmb = confusion_matrix(Yb, Pb, labels=[2,3])
print(f"\n[binary C2/C3] n={mask.sum()}  Acc={accb:.4f}  Prec(W)={prb:.4f}  Rec(W)={rcb:.4f}  F1(W)={f1b:.4f}  kappa={kappab:.4f}")
print("Confusion matrix (binary, rows/cols = C2,C3):\n", cmb)
pb, rb, f1pcb, supb = precision_recall_fscore_support(Yb, Pb, average=None, labels=[2,3], zero_division=0)
for idx, c in enumerate([2,3]):
    print(f"  C{c}: P={pb[idx]:.4f} R={rb[idx]:.4f} F1={f1pcb[idx]:.4f} support={supb[idx]}")

ValueError: Run the baseline cell first to load 'pairs' into memory!

In [2]:
# ---- rebuild pairs if not in memory (e.g. kernel restarted) ----------------
import re, pandas as pd
if "pairs" not in globals():
    ROOT = "/kaggle/input/daisee/DAiSEE"
    VIDEOS_ROOT, LABELS_ROOT = f"{ROOT}/DataSet", f"{ROOT}/Labels"
    TARGET = "Engagement"
    def cid(x): return max(re.findall(r"\d+", str(x)), key=len)
    def load_labels(p):
        df = pd.read_csv(p)
        return {cid(r["ClipID"]): int(r[TARGET]) for _, r in df.iterrows()}
    labels = {"train": load_labels(f"{LABELS_ROOT}/TrainLabels.csv"),
              "val":   load_labels(f"{LABELS_ROOT}/ValidationLabels.csv"),
              "test":  load_labels(f"{LABELS_ROOT}/TestLabels.csv")}
    def index_videos(split):
        out = {}
        for r, _, fs in os.walk(os.path.join(VIDEOS_ROOT, split)):
            for f in fs:
                if f.endswith(".avi"): out[cid(f)] = os.path.join(r, f)
        return out
    videos = {"train": index_videos("Train"),
              "val":   index_videos("Validation"),
              "test":  index_videos("Test")}
    pairs = {k: [(i, videos[k][i], labels[k][i])
                 for i in videos[k] if i in labels[k]] for k in videos}
    print("pairs rebuilt:", {k: len(v) for k, v in pairs.items()})

pairs rebuilt: {'train': 4852, 'val': 1429, 'test': 1638}


In [5]:
# =============================================================================
# GranulationCNN — VARIANT C: pretrained backbone + standard CE (no focal)
# Isolates whether minority-class recovery comes from pretraining or focal loss.
# Paste into a NEW Kaggle cell. Reuses the uint8 cache — trains in minutes.
# =============================================================================
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, cohen_kappa_score)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True

NUM_CLASSES = 4
T_FRAMES, G_GRANULES, LAMBDA = 8, 4, 0.3
BATCH, EPOCHS, LR = 16, 5, 3e-4
CACHE_DIR = "/kaggle/working/frames_cache"

# ---- rebuild cached item lists from disk ------------------------------------
def list_cache(split):
    d = os.path.join(CACHE_DIR, split)
    if "pairs" not in globals():
        raise ValueError("Run the baseline cell first to load 'pairs' into memory!")
    lab = {cid_: y for (cid_, _p, y) in pairs[split]}
    items = []
    for f in os.listdir(d):
        if f.endswith(".npy"):
            clip_id = f[:-4]
            if clip_id in lab:
                items.append((os.path.join(d, f), lab[clip_id]))
    return items

cached_c = {s: list_cache(s) for s in ["train", "val", "test"]}
print("Cached items:", {k: len(v) for k, v in cached_c.items()})

# ---- dataset ----------------------------------------------------------------
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class CachedDAiSEE(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        npy, y = self.items[idx]
        arr = np.load(npy)
        x = torch.from_numpy(arr).float().permute(0, 3, 1, 2) / 255.0
        x = (x - MEAN) / STD
        return x, torch.tensor(y, dtype=torch.long)

train_loader = DataLoader(CachedDAiSEE(cached_c["train"]), batch_size=BATCH,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(CachedDAiSEE(cached_c["val"]),   batch_size=BATCH,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(CachedDAiSEE(cached_c["test"]),  batch_size=BATCH,
                          num_workers=2, pin_memory=True)

# ---- model: pretrained, standard CE (no focal, no class weights) ------------
class GranulationCNN(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, NUM_CLASSES)
    def forward(self, x):
        B, T_, C, H, W = x.shape
        z = self.backbone(x.view(B * T_, C, H, W)).flatten(1).view(B, T_, -1)
        avg_logits = self.fc(z).mean(1)
        glogits, cons = [], 0.0
        for i in range(B):
            zi = z[i]
            assign = torch.randint(0, G_GRANULES, (T_,), device=zi.device)
            pl = []
            for g in range(G_GRANULES):
                m = assign == g
                if m.any():
                    proto = zi[m].mean(0)
                    pl.append(self.fc(proto))
                    cons += ((zi[m] - proto) ** 2).mean()
            glogits.append(torch.stack(pl).mean(0))
        return avg_logits, torch.stack(glogits), cons / B

def run_epoch(model, loader, opt=None, scaler=None):
    train = opt is not None
    model.train(train)
    tot, cor, loss_sum = 0, 0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast(enabled=(DEVICE == "cuda")):
                _, gl, cons = model(x)
                loss = F.cross_entropy(gl, y) + LAMBDA * cons   # standard CE, no weights
            if train:
                opt.zero_grad()
                if scaler:
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
            cor += (gl.argmax(1) == y).sum().item()
            tot += y.size(0); loss_sum += loss.item() * y.size(0)
    return loss_sum / tot, cor / tot

model  = GranulationCNN().to(DEVICE)
opt    = torch.optim.Adam(model.parameters(), lr=LR)
scaler = GradScaler(enabled=(DEVICE == "cuda"))
best = 0.0
for e in range(1, EPOCHS + 1):
    tr = run_epoch(model, train_loader, opt, scaler)
    va = run_epoch(model, val_loader)
    print(f"Epoch {e:02d} | TrainAcc={tr[1]:.4f} | ValAcc={va[1]:.4f}")
    if va[1] > best:
        best = va[1]; torch.save(model.state_dict(), "best_granulation_pretrained_ce.pt")
print("Best Val Acc:", best)

# ---- evaluate ---------------------------------------------------------------
model.load_state_dict(torch.load("best_granulation_pretrained_ce.pt")); model.eval()
def evaluate(model, loader):
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            _, gl, _ = model(x)
            P.append(gl.argmax(1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)
P, Y = evaluate(model, test_loader)

acc = accuracy_score(Y, P)
pr, rc, f1, _ = precision_recall_fscore_support(Y, P, average="weighted", zero_division=0)
kappa = cohen_kappa_score(Y, P)
cm = confusion_matrix(Y, P)
print(f"\n[4-class] Acc={acc:.4f}  Prec(W)={pr:.4f}  Rec(W)={rc:.4f}  F1(W)={f1:.4f}  kappa={kappa:.4f}")
print("Confusion matrix (4-class):\n", cm)
p, r, f1pc, sup = precision_recall_fscore_support(Y, P, average=None, zero_division=0)
for c in range(NUM_CLASSES):
    print(f"  C{c}: P={p[c]:.4f} R={r[c]:.4f} F1={f1pc[c]:.4f} support={sup[c]}")

# ---- binary C2/C3 view ------------------------------------------------------
mask = (Y == 2) | (Y == 3)
Yb, Pb = Y[mask], P[mask]
accb = accuracy_score(Yb, Pb)
prb, rcb, f1b, _ = precision_recall_fscore_support(Yb, Pb, average="weighted",
                                                    labels=[2,3], zero_division=0)
kappab = cohen_kappa_score(Yb, Pb)
cmb = confusion_matrix(Yb, Pb, labels=[2,3])
print(f"\n[binary C2/C3] n={int(mask.sum())}  Acc={accb:.4f}  Prec(W)={prb:.4f}  "
      f"Rec(W)={rcb:.4f}  F1(W)={f1b:.4f}  kappa={kappab:.4f}")
print("Confusion matrix (binary, rows/cols = C2,C3):\n", cmb)
pb, rb, f1pcb, supb = precision_recall_fscore_support(Yb, Pb, average=None,
                                                       labels=[2,3], zero_division=0)
for idx, c in enumerate([2, 3]):
    print(f"  C{c}: P={pb[idx]:.4f} R={rb[idx]:.4f} F1={f1pcb[idx]:.4f} support={supb[idx]}")

Cached items: {'train': 4852, 'val': 1429, 'test': 1638}
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 144MB/s] 
/tmp/ipykernel_57/4255430627.py:107: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE == "cuda"))
/tmp/ipykernel_57/4255430627.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE == "cuda")):


Epoch 01 | TrainAcc=0.5433 | ValAcc=0.5871
Epoch 02 | TrainAcc=0.5911 | ValAcc=0.3716
Epoch 03 | TrainAcc=0.6000 | ValAcc=0.5710
Epoch 04 | TrainAcc=0.6173 | ValAcc=0.5038
Epoch 05 | TrainAcc=0.6228 | ValAcc=0.4822
Best Val Acc: 0.5871238628411477

[4-class] Acc=0.5324  Prec(W)=0.4967  Rec(W)=0.5324  F1(W)=0.4232  kappa=0.0432
Confusion matrix (4-class):
 [[  0   0   3   1]
 [  0   0  62  19]
 [  0   0 805  44]
 [  0   0 637  67]]
  C0: P=0.0000 R=0.0000 F1=0.0000 support=4
  C1: P=0.0000 R=0.0000 F1=0.0000 support=81
  C2: P=0.5342 R=0.9482 F1=0.6834 support=849
  C3: P=0.5115 R=0.0952 F1=0.1605 support=704

[binary C2/C3] n=1553  Acc=0.5615  Prec(W)=0.5788  Rec(W)=0.5615  F1(W)=0.4587  kappa=0.0467
Confusion matrix (binary, rows/cols = C2,C3):
 [[805  44]
 [637  67]]
  C2: P=0.5583 R=0.9482 F1=0.7027 support=849
  C3: P=0.6036 R=0.0952 F1=0.1644 support=704


In [ ]:
# =============================================================================
# GranulationCNN — Multi-seed sweep (Variants A, B, C across 5 seeds)
# Paste into ONE Kaggle cell AFTER the baseline cell has run (cache exists).
# Runs 3 variants x 5 seeds = 15 training runs. ~30-45 min on T4.
# Prints mean ± std for all key metrics at the end.
# =============================================================================
import os, re, numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, cohen_kappa_score)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)

# ---- config -----------------------------------------------------------------
NUM_CLASSES = 4
T_FRAMES, G_GRANULES, LAMBDA = 8, 4, 0.3
BATCH, EPOCHS, LR = 16, 5, 3e-4
CACHE_DIR = "/kaggle/working/frames_cache"
SEEDS = [42, 123, 456, 789, 2024]

# ---- rebuild pairs if kernel restarted --------------------------------------
if "pairs" not in globals():
    ROOT = "/kaggle/input/daisee/DAiSEE"
    VIDEOS_ROOT, LABELS_ROOT = f"{ROOT}/DataSet", f"{ROOT}/Labels"
    TARGET = "Engagement"
    def cid(x): return max(re.findall(r"\d+", str(x)), key=len)
    def load_labels(p):
        df = pd.read_csv(p)
        return {cid(r["ClipID"]): int(r[TARGET]) for _, r in df.iterrows()}
    labels_map = {"train": load_labels(f"{LABELS_ROOT}/TrainLabels.csv"),
                  "val":   load_labels(f"{LABELS_ROOT}/ValidationLabels.csv"),
                  "test":  load_labels(f"{LABELS_ROOT}/TestLabels.csv")}
    def index_videos(split):
        out = {}
        for r, _, fs in os.walk(os.path.join(VIDEOS_ROOT, split)):
            for f in fs:
                if f.endswith(".avi"): out[cid(f)] = os.path.join(r, f)
        return out
    videos = {"train": index_videos("Train"),
              "val":   index_videos("Validation"),
              "test":  index_videos("Test")}
    pairs = {k: [(i, videos[k][i], labels_map[k][i])
                 for i in videos[k] if i in labels_map[k]] for k in videos}
    print("pairs rebuilt:", {k: len(v) for k, v in pairs.items()})

# ---- rebuild cached item lists ----------------------------------------------
def list_cache(split):
    d = os.path.join(CACHE_DIR, split)
    lab = {cid_: y for (cid_, _p, y) in pairs[split]}
    items = []
    for f in os.listdir(d):
        if f.endswith(".npy"):
            clip_id = f[:-4]
            if clip_id in lab:
                items.append((os.path.join(d, f), lab[clip_id]))
    return items

cached = {s: list_cache(s) for s in ["train", "val", "test"]}
print("cache:", {k: len(v) for k, v in cached.items()})

# ---- dataset ----------------------------------------------------------------
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class CachedDAiSEE(Dataset):
    def __init__(self, items): self.items = items
    def __len__(self): return len(self.items)
    def __getitem__(self, idx):
        npy, y = self.items[idx]
        arr = np.load(npy)
        x = torch.from_numpy(arr).float().permute(0, 3, 1, 2) / 255.0
        x = (x - MEAN) / STD
        return x, torch.tensor(y, dtype=torch.long)

# ---- class weights for focal variants ---------------------------------------
train_labels = np.array([y for _, y in cached["train"]])
counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float32)
inv = counts.sum() / (counts + 1e-6)
class_w = torch.tensor(inv / inv.sum() * NUM_CLASSES,
                        dtype=torch.float32, device=DEVICE)
print("class weights:", class_w.cpu().numpy().round(3).tolist())

# ---- focal loss -------------------------------------------------------------
def focal_loss(logits, target, gamma=2.0, weight=None):
    logp = F.log_softmax(logits, dim=1)
    p    = logp.exp()
    logp_t = logp.gather(1, target.unsqueeze(1)).squeeze(1)
    p_t    = p.gather(1, target.unsqueeze(1)).squeeze(1)
    loss   = -((1 - p_t) ** gamma) * logp_t
    if weight is not None:
        loss = loss * weight[target]
    return loss.mean()

# ---- model ------------------------------------------------------------------
def make_model(pretrained):
    base = models.resnet18(
        weights="IMAGENET1K_V1" if pretrained else None)
    backbone = nn.Sequential(*list(base.children())[:-1])
    fc = nn.Linear(512, NUM_CLASSES)

    class GranulationCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.backbone = backbone
            self.fc = fc
        def forward(self, x):
            B, T_, C, H, W = x.shape
            z = self.backbone(x.view(B*T_, C, H, W)).flatten(1).view(B, T_, -1)
            avg_logits = self.fc(z).mean(1)
            glogits, cons = [], 0.0
            for i in range(B):
                zi = z[i]
                assign = torch.randint(0, G_GRANULES, (T_,), device=zi.device)
                pl = []
                for g in range(G_GRANULES):
                    m = assign == g
                    if m.any():
                        proto = zi[m].mean(0)
                        pl.append(self.fc(proto))
                        cons += ((zi[m] - proto) ** 2).mean()
                glogits.append(torch.stack(pl).mean(0))
            return avg_logits, torch.stack(glogits), cons / B
    return GranulationCNN()

# ---- train / eval -----------------------------------------------------------
def set_seed(s):
    torch.manual_seed(s); torch.cuda.manual_seed_all(s); np.random.seed(s)

def make_loaders(seed):
    g = torch.Generator(); g.manual_seed(seed)
    tr = DataLoader(CachedDAiSEE(cached["train"]), batch_size=BATCH,
                    shuffle=True, num_workers=2, pin_memory=True, generator=g)
    va = DataLoader(CachedDAiSEE(cached["val"]),   batch_size=BATCH,
                    num_workers=2, pin_memory=True)
    te = DataLoader(CachedDAiSEE(cached["test"]),  batch_size=BATCH,
                    num_workers=2, pin_memory=True)
    return tr, va, te

def run_epoch(model, loader, opt=None, scaler=None, loss_fn="ce"):
    train = opt is not None
    model.train(train)
    tot, cor, loss_sum = 0, 0, 0.0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with autocast(enabled=(DEVICE=="cuda")):
                _, gl, cons = model(x)
                if loss_fn == "focal":
                    cls = focal_loss(gl, y, gamma=2.0, weight=class_w)
                else:
                    cls = F.cross_entropy(gl, y)
                loss = cls + LAMBDA * cons
            if train:
                opt.zero_grad()
                if scaler:
                    scaler.scale(loss).backward()
                    scaler.step(opt); scaler.update()
                else:
                    loss.backward(); opt.step()
            cor += (gl.argmax(1) == y).sum().item()
            tot += y.size(0); loss_sum += loss.item() * y.size(0)
    return loss_sum / tot, cor / tot

def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            _, gl, _ = model(x)
            P.append(gl.argmax(1).cpu().numpy()); Y.append(y.numpy())
    P, Y = np.concatenate(P), np.concatenate(Y)
    acc   = accuracy_score(Y, P)
    pr, rc, f1, _ = precision_recall_fscore_support(
        Y, P, average="weighted", zero_division=0)
    kappa = cohen_kappa_score(Y, P)
    # binary C2/C3
    mask = (Y==2)|(Y==3)
    Yb, Pb = Y[mask], P[mask]
    kappab = cohen_kappa_score(Yb, Pb) if len(np.unique(Pb)) > 1 else 0.0
    accb   = accuracy_score(Yb, Pb)
    _, _, f1b, _ = precision_recall_fscore_support(
        Yb, Pb, average="weighted", labels=[2,3], zero_division=0)
    # per-class recall
    _, rec_pc, _, _ = precision_recall_fscore_support(
        Y, P, average=None, zero_division=0)
    c1_rec = rec_pc[1] if len(rec_pc) > 1 else 0.0
    c3_rec = rec_pc[3] if len(rec_pc) > 3 else 0.0
    return dict(acc=acc, f1w=f1, kappa=kappa,
                acc_b=accb, f1b=f1b, kappa_b=kappab,
                c1_rec=c1_rec, c3_rec=c3_rec)

# ---- variant definitions ----------------------------------------------------
VARIANTS = {
    "A_random_CE":        dict(pretrained=False, loss_fn="ce"),
    "B_pretrained_focal": dict(pretrained=True,  loss_fn="focal"),
    "C_pretrained_CE":    dict(pretrained=True,  loss_fn="ce"),
}

# ---- main sweep -------------------------------------------------------------
results = {v: [] for v in VARIANTS}

for vname, vcfg in VARIANTS.items():
    print(f"\n{'='*60}")
    print(f"VARIANT: {vname}")
    print(f"{'='*60}")
    for seed in SEEDS:
        set_seed(seed)
        tr_loader, va_loader, te_loader = make_loaders(seed)
        model  = make_model(vcfg["pretrained"]).to(DEVICE)
        opt    = torch.optim.Adam(model.parameters(), lr=LR)
        scaler = GradScaler(enabled=(DEVICE=="cuda"))
        best_val, best_state = 0.0, None
        for e in range(1, EPOCHS+1):
            run_epoch(model, tr_loader, opt, scaler, vcfg["loss_fn"])
            _, va_acc = run_epoch(model, va_loader, loss_fn=vcfg["loss_fn"])
            if va_acc > best_val:
                best_val = va_acc
                best_state = {k: v.cpu().clone()
                              for k, v in model.state_dict().items()}
        model.load_state_dict({k: v.to(DEVICE)
                               for k, v in best_state.items()})
        m = evaluate(model, te_loader)
        results[vname].append(m)
        print(f"  seed={seed} | acc={m['acc']:.4f} f1={m['f1w']:.4f} "
              f"k={m['kappa']:.4f} k_b={m['kappa_b']:.4f} "
              f"c3r={m['c3_rec']:.3f}")

# ---- summary table ----------------------------------------------------------
print(f"\n{'='*70}")
print("SUMMARY: mean ± std across 5 seeds")
print(f"{'='*70}")
header = (f"{'Variant':<25} {'Acc':>10} {'F1(W)':>10} "
          f"{'kappa':>10} {'k_bin':>10} {'C3rec':>8}")
print(header)
print("-" * len(header))
for vname, runs in results.items():
    keys = ["acc","f1w","kappa","kappa_b","c3_rec"]
    vals = {k: np.array([r[k] for r in runs]) for k in keys}
    row = (f"{vname:<25} "
           f"{vals['acc'].mean():.4f}±{vals['acc'].std():.4f}  "
           f"{vals['f1w'].mean():.4f}±{vals['f1w'].std():.4f}  "
           f"{vals['kappa'].mean():.4f}±{vals['kappa'].std():.4f}  "
           f"{vals['kappa_b'].mean():.4f}±{vals['kappa_b'].std():.4f}  "
           f"{vals['c3_rec'].mean():.3f}±{vals['c3_rec'].std():.3f}")
    print(row)

print(f"\nAll raw results:")
for vname, runs in results.items():
    print(f"\n{vname}:")
    for s, r in zip(SEEDS, runs):
        print(f"  seed={s}: acc={r['acc']:.4f} f1={r['f1w']:.4f} "
              f"kappa={r['kappa']:.4f} kappa_b={r['kappa_b']:.4f} "
              f"c1_rec={r['c1_rec']:.3f} c3_rec={r['c3_rec']:.3f}")

Device: cuda
cache: {'train': 4852, 'val': 1429, 'test': 1638}
class weights: [3.3529999256134033, 0.5509999990463257, 0.04500000178813934, 0.052000001072883606]

VARIANT: A_random_CE


/tmp/ipykernel_57/3820145014.py:218: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE=="cuda"))
/tmp/ipykernel_57/3820145014.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE=="cuda")):


  seed=42 | acc=0.4939 f1=0.4265 k=-0.0168 k_b=-0.0208 c3r=0.161


/tmp/ipykernel_57/3820145014.py:218: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE=="cuda"))
/tmp/ipykernel_57/3820145014.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE=="cuda")):


  seed=123 | acc=0.5195 f1=0.3825 k=0.0087 k_b=0.0099 c3r=0.040


...

/tmp/ipykernel_57/3820145014.py:218: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE=="cuda"))
/tmp/ipykernel_57/3820145014.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE=="cuda")):


  seed=123 | acc=0.5330 f1=0.4082 k=0.0385 k_b=0.0435 c3r=0.070


/tmp/ipykernel_57/3820145014.py:218: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE=="cuda"))
/tmp/ipykernel_57/3820145014.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE=="cuda")):


  seed=456 | acc=0.5336 f1=0.4238 k=0.0462 k_b=0.0468 c3r=0.092


/tmp/ipykernel_57/3820145014.py:218: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(DEVICE=="cuda"))
/tmp/ipykernel_57/3820145014.py:152: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(DEVICE=="cuda")):


{'train': 4852, 'val': 1429, 'test': 1638}


True


Train dir sample: ['205601', '110041', '200050', '110010', '181374', '210061', '459999', '310072', '210059', '310081']
Root: /kaggle/input/daisee/DAiSEE/DataSet/Train
Dirs: ['205601', '110041', '200050', '110010', '181374']
Files: []


Index(['ClipID', 'Boredom', 'Engagement', 'Confusion', 'Frustration '], dtype='object')
           ClipID  Boredom  Engagement  Confusion  Frustration 
0  1100011002.avi        0           2          0             0
1  1100011003.avi        0           2          0             0
2  1100011004.avi        0           3          0             0
3  1100011005.avi        0           3          0             0
4  1100011006.avi        0           3          0             0


Train videos: 5482
Val videos: 1720
Test videos: 1866


{'train': 5358, 'val': 1429, 'test': 1784}


KeyboardInterrupt: 

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import torch

def evaluate_full(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            out = model(x)
            logits = out[0] if isinstance(out, tuple) else out

            preds = torch.argmax(logits, dim=1)

            all_preds.append(preds.cpu().numpy())
            all_labels.append(y.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    acc = accuracy_score(all_labels, all_preds)

    prec, rec, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="weighted",
        zero_division=0
    )

    cm = confusion_matrix(all_labels, all_preds)

    return acc, prec, rec, f1, cm, all_labels, all_preds


In [ ]:
acc, prec, rec, f1, cm, all_labels, all_preds = evaluate_full(
    model,
    test_loader,
    DEVICE
)

print(f"Test Accuracy : {acc:.4f}")
print(f"Precision     : {prec:.4f}")
print(f"Recall        : {rec:.4f}")
print(f"F1-score      : {f1:.4f}")
print("Confusion Matrix:\n", cm)


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p, r, f1_pc, support = precision_recall_fscore_support(
    all_labels,
    all_preds,
    average=None,
    zero_division=0
)

print("Per-class F1:", f1_pc)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,4))
plt.bar(range(len(f1_pc)), f1_pc)
plt.xticks(range(len(f1_pc)), ["C0","C1","C2","C3"])
plt.ylabel("F1-score")
plt.title("Per-class F1 Scores (DAiSEE)")
plt.ylim(0,1)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

true_counts = np.bincount(all_labels)
pred_counts = np.bincount(all_preds, minlength=len(true_counts))

x = np.arange(len(true_counts))

plt.figure(figsize=(7,4))
plt.bar(x - 0.2, true_counts, width=0.4, label="Ground Truth")
plt.bar(x + 0.2, pred_counts, width=0.4, label="Predicted")
plt.xticks(x, ["C0","C1","C2","C3"])
plt.ylabel("Count")
plt.title("Class Distribution: Ground Truth vs Predictions")
plt.legend()
plt.show()


In [ ]:
def save_confusion_matrix(cm, class_names):
    plt.figure(figsize=(4.5, 4))  # IEEE column-friendly
    plt.imshow(cm)
    plt.colorbar()

    plt.xticks(range(len(class_names)), class_names, rotation=45)
    plt.yticks(range(len(class_names)), class_names)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     ha="center", va="center")

    plt.xlabel("Predicted Class")
    plt.ylabel("True Class")
    plt.title("Confusion Matrix")

    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
def save_class_distribution(all_labels, all_preds, class_names):
    true_counts = np.bincount(all_labels, minlength=len(class_names))
    pred_counts = np.bincount(all_preds, minlength=len(class_names))

    x = np.arange(len(class_names))
    width = 0.35

    plt.figure(figsize=(5.5, 3.8))
    plt.bar(x - width/2, true_counts, width, label="Ground Truth")
    plt.bar(x + width/2, pred_counts, width, label="Predicted")

    plt.xticks(x, class_names)
    plt.ylabel("Number of Samples")
    plt.title("Class Distribution")
    plt.legend()

    plt.tight_layout()
    plt.savefig("class_distribution.png", dpi=300, bbox_inches="tight")
    plt.close()


In [ ]:
def save_per_class_f1(all_labels, all_preds, class_names):
    _, _, f1_pc, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average=None,
        zero_division=0
    )

    plt.figure(figsize=(4.8, 3.6))
    plt.bar(range(len(f1_pc)), f1_pc)

    plt.xticks(range(len(class_names)), class_names)
    plt.ylabel("F1-score")
    plt.ylim(0, 1)
    plt.title("Per-class F1 Score")

    plt.tight_layout()
    plt.savefig("per_class_f1.png", dpi=300, bbox_inches="tight")
    plt.close()

    return f1_pc


NameError: name 'model' is not defined

In [ ]:
for name, pred in models.items():
    cm = confusion_matrix(y_true, pred, labels=LABELS)

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{c}" for c in CLASS_NAMES],
        columns=[f"pred_{c}" for c in CLASS_NAMES]
    )

    cm_df.to_csv(f"{SAVE_DIR}/confusion_matrix_{name}.csv")

    plt.figure(figsize=(5.2, 4.6))
    plt.imshow(cm)
    plt.colorbar()

    plt.xticks(range(4), CLASS_NAMES)
    plt.yticks(range(4), CLASS_NAMES)

    for i in range(4):
        for j in range(4):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.xlabel("Predicted class")
    plt.ylabel("True class")
    plt.title(f"{name} Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"{SAVE_DIR}/confusion_matrix_{name}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
mcnemar_rows = []

mcnemar_rows.append(
    mcnemar_test(
        y_true,
        pred_gran,
        pred_avg,
        model_a="GranulationCNN",
        model_b="TemporalAvg"
    )
)

mcnemar_rows.append(
    mcnemar_test(
        y_true,
        pred_gran,
        pred_majority,
        model_a="GranulationCNN",
        model_b="Majority"
    )
)

mcnemar_rows.append(
    mcnemar_test(
        y_true,
        pred_avg,
        pred_majority,
        model_a="TemporalAvg",
        model_b="Majority"
    )
)

mcnemar_df = pd.DataFrame(mcnemar_rows)
mcnemar_df.to_csv(f"{SAVE_DIR}/mcnemar_tests.csv", index=False)

bootstrap_rows = []

for name, pred in models.items():
    for metric in ["accuracy", "weighted_f1"]:
        row = bootstrap_ci(y_true, pred, metric=metric)
        row["comparison"] = name
        row["type"] = "single_model_metric"
        bootstrap_rows.append(row)

comparisons = [
    ("GranulationCNN_minus_TemporalAvg", pred_gran, pred_avg),
    ("GranulationCNN_minus_Majority", pred_gran, pred_majority),
    ("TemporalAvg_minus_Majority", pred_avg, pred_majority)
]

for comp_name, pred_a, pred_b in comparisons:
    for metric in ["accuracy", "weighted_f1"]:
        row = bootstrap_ci(y_true, pred_a, pred_b, metric=metric)
        row["comparison"] = comp_name
        row["type"] = "paired_difference"
        bootstrap_rows.append(row)

bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df = bootstrap_df[
    ["comparison", "type", "metric", "median", "mean", "ci_low_2_5", "ci_high_97_5"]
]

bootstrap_df.to_csv(f"{SAVE_DIR}/bootstrap_confidence_intervals.csv", index=False)

print("=== MCNEMAR TESTS ===")
print(mcnemar_df)

print("\n=== BOOTSTRAP CONFIDENCE INTERVALS ===")
print(bootstrap_df)